# Gold Layer - Retail Analytics Platform

## Objective

Create business-ready aggregated datasets for reporting and analytics.

## KPIs

- Monthly Revenue
- Top Products
- Top Customers
- Country Sales
- Customer Analytics

## Source

workspace.retail.silver_online_retail

## Target

workspace.retail.gold_*

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


#Step 1: Read Silver Table
silver_df = spark.table(
    "workspace.retail.silver_online_retail"
)

In [0]:
# KPI 1: Monthly Revenue
monthly_revenue = (
    silver_df
    .withColumn(
        "year",
        F.year("invoice_timestamp")
    )
    .withColumn(
        "month",
        F.month("invoice_timestamp")
    )
    .groupBy(
        "year",
        "month"
    )
    .agg(
        F.round(
            F.sum("revenue"),
            2
        ).alias("total_revenue")
    )
    .orderBy(
        "year",
        "month"
    )
)

display(monthly_revenue)

year,month,total_revenue
2009,12,683504.01
2010,1,555802.67
2010,2,504558.95
2010,3,696978.47
2010,4,591982.0
2010,5,597833.38
2010,6,636371.13
2010,7,589736.17
2010,8,602224.6
2010,9,829013.95


In [0]:
# KPI 2: Top Products
top_products = (
    silver_df
    .groupBy(
        "StockCode",
        "Description"
    )
    .agg(
        F.round(
            F.sum("revenue"),
            2
        ).alias("total_revenue")
    )
)


StockCode,Description,total_revenue
22353,LUNCHBOX WITH CUTLERY FAIRY CAKES,3215.25
21351,CINAMMON & ORANGE WREATH,3018.7
20695,FLORAL BLUE MONSTER,332.0
21033,JUMBO BAG CHARLIE AND LOLA TOYS,6463.67
84031A,CHARLIE+LOLA RED HOT WATER BOTTLE,2223.32
85132C,CHARLIE AND LOLA FIGURES TINS,4843.53
22138,BAKING SET 9 PIECE RETROSPOT,43070.91
22272,FELTCRAFT DOLL MARIA,3684.91
21916,SET 12 RETRO WHITE CHALK STICKS,4211.98
22253,6 CROCHET STRAWBERRIES,1096.39


In [0]:
# Product Ranking

# Window Function

window_spec = Window.orderBy(
    F.desc("total_revenue")
)

top_products = (
    top_products
    .withColumn(
        "product_rank",
        F.dense_rank()
        .over(window_spec)
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
#Save

top_products.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(
"workspace.retail.gold_top_products"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
#KPI 3: Top Customers
top_customers = (
    silver_df
    .groupBy(
        "Customer_ID"
    )
    .agg(
        F.round(
            F.sum("revenue"),
            2
        ).alias("customer_revenue")
    )
)

#Customer Ranking

customer_window = Window.orderBy(
    F.desc("customer_revenue")
)

top_customers = (
    top_customers
    .withColumn(
        "customer_rank",
        F.dense_rank()
        .over(customer_window)
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
#Save
top_customers.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(
"workspace.retail.gold_top_customers"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# KPI 4: Country Sales
country_sales = (
    silver_df
    .groupBy("Country")
    .agg(
        F.round(
            F.sum("revenue"),
            2
        ).alias("country_revenue")
    )
)
#save
country_sales.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(
"workspace.retail.gold_country_sales"
)



In [0]:
#KPI 5: Daily Revenue Trend
daily_sales = (
    silver_df
    .groupBy(
        F.to_date(
            "invoice_timestamp"
        ).alias("sales_date")
    )
    .agg(
        F.round(
            F.sum("revenue"),
            2
        ).alias("daily_revenue")
    )
)
#save
daily_sales.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(
"workspace.retail.gold_daily_sales"
)

In [0]:
#Validation
#Check Gold Tables
spark.sql("""
SHOW TABLES IN workspace.retail
""").show(truncate=False)

+--------+--------------------+-----------+
|database|tableName           |isTemporary|
+--------+--------------------+-----------+
|retail  |bronze_online_retail|false      |
|retail  |gold_country_sales  |false      |
|retail  |gold_top_customers  |false      |
|retail  |gold_top_products   |false      |
|retail  |silver_online_retail|false      |
+--------+--------------------+-----------+

